# Train Multi-Armed Bandits

A multi-armed bandit learns one expected reward per action without state transitions. This notebook uses a five-arm Gaussian bandit with epsilon-greedy exploration.

## Incremental estimate

$$Q_{n+1}(a)=Q_n(a)+\frac{1}{N_{n+1}(a)}\left[R_{n+1}-Q_n(a)\right].$$

Here $Q_n(a)$ is arm $a$'s estimate after $n$ pulls, $R_{n+1}$ the new reward, and $N_{n+1}(a)$ the arm's updated pull count.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from aprenderl import MultiArmedBandit, MultiArmedBanditConfig
from aprenderl.utils import evaluate_policy


class GaussianBandit(gym.Env[int, int]):
    observation_space = gym.spaces.Discrete(1)
    action_space = gym.spaces.Discrete(5)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        return 0, {}

    def step(self, action):
        means = np.arange(5, dtype=np.float32) * 0.25
        reward = self.np_random.normal(means[action], 1.0)
        return 0, float(reward), True, False, {}


In [ ]:
env = GaussianBandit()
config = MultiArmedBanditConfig(
    exploration_initial_epsilon=0.2,
    exploration_final_epsilon=0.05,
    exploration_steps=2_000,
    seed=7,
)

agent = MultiArmedBandit(env, config=config)
agent.learn(total_timesteps=5_000, progress_bar=False)
print("Estimates:", agent.q_values)
print("Pull counts:", agent.action_counts)
env.close()

In [ ]:
plt.bar(np.arange(agent.action_dim), agent.q_values)
plt.xlabel("Arm")
plt.ylabel("Estimated reward")
plt.title("Learned arm values")
plt.grid(axis="y", alpha=0.2)
plt.show()

## Evaluate the learned arm

A bandit has no visual state to render, so this runs 1,000 independent pulls using the deterministic best arm.

In [ ]:
evaluation_env = GaussianBandit()
try:
    result = evaluate_policy(
        agent, evaluation_env, episodes=1_000, deterministic=True, seed=10
    )
finally:
    evaluation_env.close()

print(f"Mean reward: {result.mean_return:.2f} +/- {result.return_std:.2f}")